# LoRA

Fine-tuning adapts a pretrained model to new tasks by updating its weights. But updating *all* the weights in a large model is expensive — both in memory and compute. **LoRA (Low-Rank Adaptation)** is a technique that makes fine-tuning much cheaper by exploiting the fact that weight updates during fine-tuning (as opposed to pretraining) tend to be *low rank*.

In this activity, you'll build up the intuition behind LoRA step by step:
1. Review matrix rank and SVD to understand what "low rank" means
2. Compare full-rank and low-rank linear layers on a synthetic task
3. Implement a LoRA layer and see why it works

In [ ]:
import torch
from torch import nn
import torch.nn.functional as F
from tqdm import tqdm

## Connecting back to SVD

Every matrix has a **rank** — the number of linearly independent rows (or columns). A matrix with rank $r$ can be perfectly decomposed into a product of two smaller matrices with inner dimension $r$.

The **Singular Value Decomposition (SVD)**, which you may have already seen before elsewhere, gives us exactly this. For any matrix $A \in \mathbb{R}^{m \times n}$:

$$A = U \Sigma V^T$$

where $U$ is $m \times m$, $\Sigma$ is diagonal with singular values, and $V^T$ is $n \times n$.

If we keep only the top $r$ singular values (setting the rest to zero), we get the best rank-$r$ approximation of $A$. The smaller singular values contribute less to the matrix, so we can often approximate a full-rank matrix reasonably well with a low-rank one.

Run the cells below to see this in action. Notice how even with rank 4 (half the full rank of 8), we can still reconstruct a decent approximation of the matrix.

In [ ]:
A = torch.randn(16, 8)
A

In [ ]:
torch.linalg.matrix_rank(A)

In [ ]:
U, S, Vt = torch.linalg.svd(A)
print(f"U shape: {U.shape}\nS shape: {S.shape}\nVt shape: {Vt.shape}")

r = 4
A_approx = (U[:, :r] * S[:r]) @ Vt[:r, :]
error = torch.norm(A - A_approx) / torch.norm(A)
print(f"Reconstruction error of A with rank {r}: {error:.4f}")

## Low Rank Factorization

Tody's neural networks (including transformers) are composed of many large, dense weight matrices. Instead of storing a full $d_{out} \times d_{in}$ weight matrix $W$, we can represent it as a product of two smaller matrices:

$$W = B A$$

where $A \in \mathbb{R}^{r \times d_{in}}$ and $B \in \mathbb{R}^{d_{out} \times r}$, with $r \ll \min(d_{in}, d_{out})$.

This reduces the number of parameters from $d_{out} \cdot d_{in}$ to $r \cdot (d_{in} + d_{out})$. The trade-off is that $W = BA$ is constrained to have rank at most $r$, so it can't represent every possible linear transformation — only those that live in a low-dimensional subspace.

Run the cell below to see the parameter savings for a concrete example.

In [ ]:
d_in = 64
d_out = 32
r = 8  # rank of the factorization

# Full weight matrix
W_full = torch.randn(d_out, d_in)

# Low-rank factorization: two smaller matrices
A = torch.randn(r, d_in)    
B = torch.randn(d_out, r)  

# Reconstruct the matrix from the factors
W_lowrank = B @ A

print(f"W_full shape:    {W_full.shape}  — parameters: {W_full.numel()}")
print(f"A shape:         {A.shape}       — parameters: {A.numel()}")
print(f"B shape:         {B.shape}       — parameters: {B.numel()}")
print(f"W_lowrank shape: {W_lowrank.shape}  — parameters: {A.numel() + B.numel()}")
print()
print(f"Full matrix parameters:     {W_full.numel()}")
print(f"Low-rank total parameters:  {A.numel() + B.numel()}")

## Comparing Full and Low Rank Linear Layers on Synthetic Data

Let's see what happens when we try to *learn* a full-rank transformation using a low-rank layer. We'll generate synthetic data where the true mapping is a full-rank matrix $W_{full}$, then train both a full-rank and a low-rank layer to recover it.

The full-rank layer should have no trouble — it has enough capacity to represent any linear map. But the low-rank layer is constrained: it can only learn transformations of rank $r$. If the true transformation has rank greater than $r$, the low-rank layer will never perfectly fit the data.

Run the cells below to train both models and compare their losses.

In [ ]:
# Create random synthetic data and labels 
# Our labels are just a random linear transformation 
inputs = torch.randn(10000, d_in)
labels = inputs @ W_full.T 

In [ ]:
# Build a simple linear layer with no bias
class SimpleLinear(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_out, d_in))

    def forward(self, x):
        return x @ self.W.T

# Build a simple low-rank linear layer with no bias
class LowRankLinear(nn.Module):
    def __init__(self, d_in, d_out, r): 
        super().__init__()
        self.A = nn.Parameter(torch.randn(r, d_in))
        self.B = nn.Parameter(torch.randn(d_out, r))

    def forward(self, x):
        W = self.B @ self.A 
        return x @ W.T

In [ ]:
import matplotlib.pyplot as plt 

def train(model, X, Y, epochs=16, batch_size=64, lr=.01):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []

    for epoch in tqdm(range(epochs)):
        for i in range(0, X.shape[0], batch_size):
            optimizer.zero_grad()
            batch_x = X[i:i+batch_size]
            batch_y = Y[i:i+batch_size]
            outputs = model(batch_x)
            loss = F.mse_loss(outputs, batch_y)
            loss.backward()
            optimizer.step()
            # Log our loss for later
            losses.append(loss.item())
            
    return losses

def plot_losses(losses, title):
    plt.figure(figsize=(8, 4))
    plt.plot(losses, linewidth=1.5)
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(title)
    plt.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.show()
    
full_model = SimpleLinear(d_in, d_out)
low_rank_model = LowRankLinear(d_in, d_out, r)

print("Training full model...")
full_losses = train(full_model, inputs, labels)
print("Training low rank model...")
low_rank_losses = train(low_rank_model, inputs, labels)

In [ ]:
plot_losses(full_losses, 'Training Loss for Full Layer')

In [ ]:
plot_losses(low_rank_losses, 'Training Loss for Low Rank Layer')

In [ ]:
print('Full layer loss at end of training: ', full_losses[-1])

In [ ]:
print('Low rank layer loss at end of training: ', low_rank_losses[-1])

In [ ]:
# Visualize the singular values of the learned W to confirm it's full rank
U, S, Vt = torch.linalg.svd(full_model.W.data)

plt.figure(figsize=(8, 4))
plt.bar(range(len(S)), S.numpy())
plt.xlabel("Singular value index")
plt.ylabel("Magnitude")
plt.title("Singular values of learned W (full model)")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"Rank of W: {torch.linalg.matrix_rank(full_model.W.data).item()}")

## Low Rank Adaptation

We already have a good pretrained weight matrix $W$. When we fine-tune, we're learning some update $\Delta W$ such that the new weights are $W + \Delta W$. The LoRA hypothesis is that this update $\Delta W$ is low rank — even when $W$ itself is full rank.

So instead of learning a full $\Delta W$, we learn $\Delta W = BA$ where $B \in \mathbb{R}^{d_{out} \times r}$ and $A \in \mathbb{R}^{r \times d_{in}}$. During the forward pass:

$$h = (W + \frac{\alpha}{r} BA) x$$

The frozen pretrained weights $W$ handle the bulk of the computation, and the small trainable matrices $A$ and $B$ learn the task-specific adjustment. The scaling factor $\alpha / r$ controls the magnitude of the update.

In the cell below, we simulate this: we create new labels using $W_{full} + \Delta W$ where $\Delta W$ is a rank-8 perturbation, then train a LoRA layer to recover it.

In [ ]:
r_delta =  8
delta_A = torch.randn(r_delta, d_in) * 0.1
delta_B = torch.randn(d_out, r_delta) * 0.1

delta = delta_B @ delta_A 
W_ft = W_full + delta
new_labels = inputs @ W_ft.T

In [ ]:
class LoRALayer(nn.Module):
    def __init__(self, pretrained_W, r, alpha=1.0):
        super().__init__()
        d_out, d_in = pretrained_W.shape
        self.W_frozen = pretrained_W.detach().clone()
        self.W_frozen.requires_grad_(False)
        self.A = nn.Parameter(torch.randn(r, d_in) * 0.01)
        self.B = nn.Parameter(torch.zeros(d_out, r))
        self.scaling = alpha / r
    def forward(self, x):
        W = self.W_frozen + self.scaling * (self.B @ self.A)
        return x @ W.T

In [ ]:
r = 8
lora_model = LoRALayer(full_model.W, r, alpha=r)

print("Training LoRA model with pretrained W...")
lora_losses = train(lora_model, inputs, new_labels)

In [ ]:
plot_losses(lora_losses, 'Training Loss for LoRA Layer')

In [ ]:
print('LoRA layer loss at end of training: ', lora_losses[-1])

In [ ]:
# If we fine-tune the full-rank layer on the
# new data, is the resulting weight update actually low rank?

import copy

# Make a copy of the trained full model and continue training on the new labels
full_model_ft = SimpleLinear(d_in, d_out)
full_model_ft.W = nn.Parameter(full_model.W.detach().clone())

print("Fine-tuning full model on new data...")
ft_losses = train(full_model_ft, inputs, new_labels, epochs=16)
print(f"Final loss: {ft_losses[-1]:.6f}")

# Compute the weight update
delta_W = full_model_ft.W.data - full_model.W.data

# Check the rank of the update
U, S, Vt = torch.linalg.svd(delta_W)

# Plot singular values to visualize the rank structure
plt.figure(figsize=(8, 4))
plt.bar(range(len(S)), S.numpy())
plt.xlabel("Singular value index")
plt.ylabel("Magnitude")
plt.title("Singular values of the weight update (delta_W)")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

print(f"True rank of the update: {r_delta}")

## Effect of Rank on LoRA

How much does the choice of rank $r$ matter? Let's sweep over several values and see how final loss changes. We'll use the same synthetic setup: the true update has rank 8, so LoRA layers with $r \geq 8$ should be able to recover it perfectly, while lower ranks will plateau.

In [ ]:
ranks = [1, 2, 4, 8, 16, 32]
final_losses = []

for r in ranks:
    model = LoRALayer(full_model.W, r, alpha=r)
    losses = train(model, inputs, new_labels, epochs=16)
    final_losses.append(losses[-1])
    print(f"r={r:>2d}  final loss: {losses[-1]:.6f}")

plt.figure(figsize=(8, 4))
plt.plot(ranks, final_losses, marker='o', linewidth=2)
plt.xlabel("Rank (r)")
plt.ylabel("Final Loss")
plt.title("LoRA Final Loss vs. Rank")
plt.yscale("log")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

## LoRA on a Real Model

Now let's apply LoRA to an actual language model. We'll fine-tune GPT-2 on a small dataset using the `peft` library, which makes it easy to wrap any Hugging Face model with LoRA adapters.

The basic steps are:
1. Load a pretrained model
2. Wrap it with a LoRA config — this freezes the base weights and injects small trainable $A$ and $B$ matrices into the layers you specify
3. Fine-tune where only the LoRA parameters get updated
4. Compare the number of trainable parameters to the full model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

model_name = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(model_name)

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=8, 
    # All available modules for GPT-2
    target_modules=["c_proj", "c_attn", "c_attn", "c_fc"],
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

### Load and Tokenize a Dataset

We'll use [Tiny Shakespeare](https://huggingface.co/datasets/Trelis/tiny-shakespeare) — passages from Shakespeare's plays, split into chunks of up to 1024 tokens. Shakespeare's Early Modern English style is very different from the modern web text GPT-2 was trained on, so even a small amount of LoRA fine-tuning should produce a visible shift in the model's output.

In [ ]:
dataset = load_dataset("Trelis/tiny-shakespeare", split="train")

block_size = 128

def tokenize(examples):
    tokens = tokenizer(examples["Text"], truncation=False)
    # Concatenate all tokens and split into fixed-length chunks
    all_ids = [id for ids in tokens["input_ids"] for id in ids]
    chunks = [all_ids[i:i + block_size] for i in range(0, len(all_ids) - block_size, block_size)]
    return {"input_ids": chunks, "labels": chunks}

tokenized = dataset.map(tokenize, batched=True, remove_columns=dataset.column_names)
print(f"{len(tokenized)} training chunks of length {block_size}")

### Fine-Tune with LoRA

We'll train for just one epoch. Even with this tiny amount of training, you should see the model's loss decrease as it adapts to the Wikipedia text style.

In [ ]:
training_args = TrainingArguments(
    output_dir="./lora-gpt2-shakespeare",
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=5e-4,
    logging_steps=50,
    save_strategy="no",
    report_to="none",
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=tokenized,
)

trainer.train()

### Compare Generations

Let's give the same prompt to the base GPT-2 and the LoRA-fine-tuned version and see how the outputs differ.

In [ ]:
def generate(model, prompt, max_new_tokens=200):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.eval()
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.3,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

prompt = "The king said"

print("=== Base GPT-2 ===")
base_gpt2 = AutoModelForCausalLM.from_pretrained("gpt2")
print(generate(base_gpt2, prompt))

print("\n=== LoRA Fine-Tuned ===")
print(generate(lora_model, prompt))